In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I found this course late — can I still enroll and follow along?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enroll follow along course late can I still enroll"}', call_id='call_Fn5ksKja4ejLJKnwlpVQjDdu', name='search', type='function_call', id='fc_02a8992ff178274c006a53310b94f481a1b7f3f78bcfcb13ae', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Fn5ksKja4ejLJKnwlpVQjDdu',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u20

In [10]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [11]:
extract_tool_calls(result.all_messages)

[{'name': 'search',
  'arguments': '{"query":"late enroll follow along course late can I still enroll"}'}]

In [12]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late enroll follow along course late can I still enroll"}'}]

In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]


In [14]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I found this course late — can I still enroll and follow along?',
 'answer_agent': 'Yes — you can still join and follow along. The course materials, videos, and GitHub resources are available, so you can start whenever you want.\n\nOne important note: if you want a certificate, you need to finish with the live cohort and submit your project while submissions are still being accepted. Certificates aren’t awarded for fully self-paced participation.\n\n',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late enroll follow along course late can I still enroll"}'}],
 'cost': Decimal('0.00119175'),
 'document': '74eb249bbf'}

In [15]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:10], generate_agent_answer)

  0%|          | 0/10 [00:00<?, ?it/s]

In [17]:
df_agent = pd.DataFrame(agent_answers)

In [18]:
df_agent

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I found this course late — can I still enroll ...,Yes — you can still join the course and follow...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.0011340,74eb249bbf
1,Is it too late to join the course if I only di...,Yes — it’s still okay to join the course even ...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""to...",0.00101025,74eb249bbf
2,Can I start the course after it has already be...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""st...",0.0013485,74eb249bbf
3,"If I join late, am I still eligible for the ce...","Yes — you can still join late, but to be eligi...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.00090375,74eb249bbf
4,What do I need to do to get a certificate if I...,"If you’re joining now, you can still get a cer...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""ce...",0.0013245,74eb249bbf
5,"I signed up for the LLM Zoomcamp, but I still ...","Yes — that’s normal.\n\nFor LLM Zoomcamp, you ...",You don't need it. You're accepted. You can al...,"[{'name': 'search', 'arguments': '{""query"":""LL...",0.0014415,977bf7786c
6,Do I actually need a confirmation email before...,No — you don’t need to wait for the confirmati...,You don't need it. You're accepted. You can al...,"[{'name': 'search', 'arguments': '{""query"":""co...",0.00117825,977bf7786c
7,"If I registered for the LLM Zoomcamp, does tha...","Yes — for LLM Zoomcamp, registering does **not...",You don't need it. You're accepted. You can al...,"[{'name': 'search', 'arguments': '{""query"":""re...",0.0014115,977bf7786c
8,"Is registration mandatory for the course, or c...",Registration is **not mandatory**. You can **s...,You don't need it. You're accepted. You can al...,"[{'name': 'search', 'arguments': '{""query"":""re...",0.00121875,977bf7786c
9,Does the course check whether I’m on some regi...,"No — for this course, homework submissions are...",You don't need it. You're accepted. You can al...,"[{'name': 'search', 'arguments': '{""query"":""re...",0.00102375,977bf7786c


In [19]:
df_agent["cost"].sum()

Decimal('0.01199475')

In [20]:
df_agent.to_csv("data/agent-answers_partial.csv", index=False)

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [22]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [23]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [24]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent answer matches the ground truth. It correctly says the student can still join and follow along, and it includes the key caveat that to receive a certificate, the project must be submitted while submissions are still open. This is equivalent to the original answer.', answer_score='good', trajectory_reasoning='The single search query is relevant to the question and includes key ideas like late enrollment and whether the student can still enroll/follow along. One search is sufficient here, and there were no unnecessary duplicate calls.', trajectory_score='good')

In [25]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [26]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers[:10], judge_agent_record)

  0%|          | 0/10 [00:00<?, ?it/s]

In [27]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [28]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [29]:
calc_total_price(usages)

0.01006575

In [30]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    9
bad     1
Name: count, dtype: int64

In [31]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    10
Name: count, dtype: int64

In [32]:
df_agent_eval.to_csv("data/agent-evaluations_partial.csv", index=False)